In [11]:
import torch
from ultralytics import YOLO
import ultralytics.nn.tasks as tasks
import ultralytics.nn.modules as modules
from dsconv import DySnakeConv


In [12]:
device = 0 if torch.cuda.is_available() else "cpu" 
print(f"Kullanilan cihaz: {'GPU (cuda:0)' if device == 0 else 'CPU'}")
if device == "cpu":
        print("UYARI: GPU bulunamadi, egitim CPU'da yapilacak ve cok yavas olabilir.")

Kullanilan cihaz: GPU (cuda:0)


In [13]:
tasks.DySnakeConv = DySnakeConv
modules.DySnakeConv = DySnakeConv
if hasattr(modules, "__all__"):
    modules.__all__ = tuple(modules.__all__) + ("DySnakeConv",)

In [14]:
# 1) P2 head'li YOLO26 mimarisini (henüz eğitilmemiş yapı) oluştur
#    "s" ölçeğini baseline ile aynı tut ki ağırlık transferi daha sağlıklı olsun
model = YOLO("dsconv.yaml").load("../baseline/runs/detect/train/weights/best.pt")  # kendi baseline yolunu yaz

# 2) Baseline eğitiminden çıkan ağırlıkları yükle
#    .load() ismi/shape'i eşleşen katmanları (backbone, P3-P4-P5) otomatik aktarır,
#    yeni eklenen P2 katmanları rastgele başlatılmış olarak kalır
#model.load("runs/detect/train/weights/best.pt")  # kendi baseline yolunu yaz

WARNING no model scale passed. Assuming scale='n'.
Transferred 348/742 items from pretrained weights


In [15]:
# --- Aşama A: Backbone dondurulmuş, yeni katmanlar + head ısınıyor ---
model.train(
    data="../../../dataset2/yolo26/YOLO.v1i.yolo26/data.yaml",
    epochs=500,
    imgsz=640,
    batch=16,
    optimizer="SGD",
    device=device,
    name="ablation_dsconv",
)

New https://pypi.org/project/ultralytics/8.4.120 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.118  Python-3.12.9 torch-2.13.0+cu130 CUDA:0 (NVIDIA GeForce RTX 4060, 8188MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=False, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=../../../YOLO.v1i.yolo26/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=500, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mix

RuntimeError: Input type (float) and bias type (struct c10::Half) should be the same

In [16]:
from ultralytics import YOLO
from pathlib import Path
import torch

model_path = Path("runs/detect/ablation_dsconv/weights/best.pt")
if model_path.exists():
    model = YOLO(str(model_path))
    print(f"Test ediliyor: {model_path}")

    # Fuse edip özet bilgiyi ekrana bastırır (Model summary (fused): ...)
    model.fuse()
    model.info(verbose=True, imgsz=640)

    results = model.val(
        data="../../../dataset2/yolo26/YOLO.v1i.yolo26/data.yaml",
        split="test",
        imgsz=640,
        device=0 if torch.cuda.is_available() else "cpu",
    )

    metrics = results.results_dict
    precision = metrics.get("metrics/precision(B)", None)
    recall = metrics.get("metrics/recall(B)", None)
    map50 = metrics.get("metrics/mAP50(B)", None)
    map5095 = metrics.get("metrics/mAP50-95(B)", None)

    print("\nTest metrikleri:")
    print(f"Precision: {precision * 100:.2f}%" if precision is not None else "Precision: bulunamadı")
    print(f"Recall: {recall * 100:.2f}%" if recall is not None else "Recall: bulunamadı")
    print(f"mAP@0.5: {map50 * 100:.2f}%" if map50 is not None else "mAP@0.5: bulunamadı")
    print(f"mAP@0.5-0.95: {map5095 * 100:.2f}%" if map5095 is not None else "mAP@0.5-0.95: bulunamadı")
else:
    print(f"Model bulunamadı: {model_path}")

Test ediliyor: runs\detect\ablation_dsconv\weights\best.pt
dsconv summary (fused): 138 layers, 2,456,542 parameters, 0 gradients, 6.4 GFLOPs
dsconv summary (fused): 138 layers, 2,456,542 parameters, 0 gradients, 6.4 GFLOPs
Ultralytics 8.4.118  Python-3.12.9 torch-2.13.0+cu130 CUDA:0 (NVIDIA GeForce RTX 4060, 8188MiB)
val: Fast image access  (ping: 0.10.1 ms, read: 51.517.8 MB/s, size: 26.7 KB)
val: Scanning C:\Users\ertug\Desktop\yolo\YOLO.v1i.yolov11\test\labels.cache... 279 images, 0 backgrounds, 10 corrupt: 100% ━━━━━━━━━━━━ 279/279  0.0s
val: C:\Users\ertug\Desktop\yolo\YOLO.v1i.yolov11\test\images\im171_jpg.rf.e4ef6151dffe621e681fe4ca6f8e2f1b.jpg: ignoring corrupt image/label: labels mix segment and detection rows
val: C:\Users\ertug\Desktop\yolo\YOLO.v1i.yolov11\test\images\im227_jpg.rf.5070aa166043d7744a73e2b8ff14b0b4.jpg: ignoring corrupt image/label: labels mix segment and detection rows
val: C:\Users\ertug\Desktop\yolo\YOLO.v1i.yolov11\test\images\im227_jpg.rf.88d4ecb5fc66b99